# 6.1 Klasifikasi Sawah vs Non-Sawah (Sentinel-2A Jabon)

**Supervised learning** (berbeda dari clustering Bab 5 yang unsupervised): model belajar dari label sampel.

**Data & wilayah: Kecamatan Jabon** (bukan Kamal). Training memakai `data/spasial/jabon_sawah.shp` (100 poligon: 50 `sawah` + 50 `non_sawah`, CRS EPSG:32749) — turunan yang dinormalisasi dari `data/digitasi_kamal/sample_polygon_100_jabon.shp` (skema id `S001..S050`/`N001..N050`).

**PROVENANSI JUJUR:** sampel ini kandidat semi-otomatis (K-Means 8 klaster pada 11 fitur band+indeks+tekstur, lalu pelabelan klaster oleh analis dari profil spektral — lihat `data/digitasi_kamal/klasifikasi_jabon.py`), BUKAN digitasi manual QGIS + survei lapangan. Skor tinggi pada test set mencerminkan sirkularitas itu, bukan akurasi lapangan. Air/tambak (26,6% area peta 4-kelas) TIDAK disampel sehingga peta biner tidak dapat dipercaya pada area tersebut.

In [1]:
import os
import numpy as np
import pandas as pd

SEED = 42
SHP = "../data/spasial/jabon_sawah.shp"
RASTER = "../data/spasial/s2_jabon.tif"
OUT = "../data/processed"
IMG = "../data/images/eksperimenjabon/"
os.makedirs(OUT, exist_ok=True)
os.makedirs(IMG, exist_ok=True)

DATA_OK = os.path.exists(SHP) and os.path.exists(RASTER)
print("shapefile:", "ADA" if os.path.exists(SHP) else "BELUM TERSEDIA")
print("raster:", "ADA" if os.path.exists(RASTER) else "BELUM TERSEDIA")


shapefile: ADA
raster: ADA


## Data yang dipakai (inventaris lengkap)

| Berkas | Isi | Peran |
| :--- | :--- | :--- |
| `data/spasial/s2_jabon.tif` | Komposit median Sentinel-2 L2A 5–26 Agu 2026, 6 band (B02/B03/B04/B08/B11/B12), 1947x939 piksel @10 m, EPSG:32749, int16, nodata -32768 (8,3 MB) | Sumber piksel citra + sumber fitur |
| `data/spasial/jabon_sawah.shp` (+ sidecar) | 100 poligon sampel (50 sawah + 50 non_sawah), kolom `id` unik + `kelas` + `luas` (m²), EPSG:32749 | Label training |
| `data/digitasi_kamal/klasifikasi_jabon.tif` | Peta acuan 4-kelas (0=nodata, 1=sawah, 2=vegetasi, 3=terbangun, 4=air) | Pembanding peta biner |
| `data/digitasi_kamal/sample_polygon_100_jabon.shp` | Induk sampel sebelum normalisasi id | Provenansi |
| `data/digitasi_kamal/klasifikasi_jabon.py` | Skrip K-Means → pelabelan analis → sampling (seed 42) | Reproduktibilitas sampel |


## Definisi kelas (4 detail → 2 biner)

| Kode | Nama detail | Nasib biner | Kuota sampel | Ciri spektral acuan (NDVI) |
| :-: | :--- | :---: | :---: | :--- |
| 1 | Sawah (hamparan bera + tegalan halus) | **sawah** | 50 | 0,28–0,36, NIR halus |
| 2 | Pohon/Kebun/Vegetasi lain | non-sawah | 25 | 0,45–0,79 |
| 3 | Terbangun/Lahan terbuka | non-sawah | 25 | terang, NDVI ≈ 0–0,13 |
| 4 | Air/Tergenang (tambak/laut, 26,6% area!) | **tidak disampel** | 0 | NDVI < 0, NDWI > 0 |

Konsekuensi jujur: model tidak pernah melihat air/tambak saat training — piksel tambak/laut pada peta biner adalah tebakan liar model, abaikan.

## Dokumentasi fitur (7 kolom per poligon)

| Fitur | Band Sentinel-2 | Panjang gelombang / resolusi | Makna fisik untuk padi |
| :--- | :--- | :--- | :--- |
| `band_1` (B02) | Biru | 490 nm / 10 m | Hamburan atmosfer; latar air petakan |
| `band_2` (B03) | Hijau | 560 nm / 10 m | Puncak vigor vegetasi |
| `band_3` (B04) | Merah | 665 nm / 10 m | Serapan klorofil (daun aktif = gelap) |
| `band_4` (B08) | NIR | 842 nm / 10 m | Struktur sel/biomassa — pembeda utama sawah |
| `band_5` (B11) | SWIR1 | 1610 nm / 20 m→10 m | Kelembapan tanah/tanaman; terbangun |
| `band_6` (B12) | SWIR2 | 2190 nm / 20 m→10 m | Kelembapan + tanah terbuka |
| NDVI (turunan) | (B08−B04)/(B08+B04) | — | Kehijauan; dipakai di EDA + peta NDVI (bukan fitur latih) |

Ekstraksi: **mean tiap band di dalam poligon** (DN int16 → float; piksel nodata dikecualikan). Model latih memakai `band_1..band_6` saja; NDVI hanya alat diagnostik.

In [2]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask

if not DATA_OK:
    print("STATUS: BELUM TERSEDIA — pipeline dilewati.")
    df_feat = None
else:
    shp = gpd.read_file(SHP)
    shp["kelas"] = shp["kelas"].astype(str).str.strip().str.lower().str.replace("-", "_")
    _vals = set(shp["kelas"].unique())
    assert _vals <= {"sawah", "non_sawah"}, "nilai kelas tak valid"
    assert shp["id"].is_unique, "kolom id tidak unik!"
    assert shp.geometry.is_valid.all() and (~shp.geometry.is_empty).all()
    src = rasterio.open(RASTER)
    if str(shp.crs) != str(src.crs):
        shp = shp.to_crs(src.crs)
        print("CRS shapefile direproyeksi ke CRS raster:", src.crs)
    nodata = src.nodata
    rows = []
    for _, r in shp.iterrows():
        img, _ = mask(src, [r.geometry], crop=True)
        img = img.astype(float)
        img[img == nodata] = np.nan
        means = [float(np.nanmean(img[b])) for b in range(src.count)]
        rows.append({"id": r["id"], "kelas": r["kelas"],
                     **{"band_{}".format(i + 1): v for i, v in enumerate(means)}})
    df_feat = pd.DataFrame(rows).dropna()
    print(df_feat["kelas"].value_counts().to_dict(), "sampel:", len(df_feat))
    print("matriks latih: {} baris x {} kolom (id + kelas + 6 band)".format(*df_feat.shape))
    print("missing total:", int(df_feat.isna().sum().sum()), "| id unik:", bool(df_feat["id"].is_unique))
    print("rata-rata band per kelas (DN):")
    _bc = df_feat.filter(like="band_").columns.tolist()
    print(df_feat.groupby("kelas")[_bc].mean().round(1).to_string())
    df_feat.to_csv(OUT + "/jabon_features.csv", index=False)
    print("tersimpan: data/processed/jabon_features.csv")


{'non_sawah': 50, 'sawah': 50} sampel: 100
matriks latih: 100 baris x 8 kolom (id + kelas + 6 band)
missing total: 0 | id unik: True
rata-rata band per kelas (DN):
           band_1  band_2  band_3  band_4  band_5  band_6
kelas                                                    
non_sawah  1149.4  1379.1  1426.4  2637.4  2299.0  1839.7
sawah       397.4   593.7   546.5  1080.3   937.5   717.7
tersimpan: data/processed/jabon_features.csv


## Split, training, evaluasi

Split stratified 80/20 (seed 42). Model: RandomForest (300 pohon, seed 42). Metrik: confusion matrix, accuracy, precision, recall, F1 + CV 5-fold. Keterbatasan: label kandidat sirkular (skor bagus ≠ akurasi lapangan), tanpa sampel air/tambak, tetangga spasial train/test bisa berdekatan.

In [3]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, classification_report

if df_feat is None:
    print("STATUS: BELUM TERSEDIA — training dilewati.")
else:
    X = df_feat.filter(like="band_").to_numpy()
    y = (df_feat["kelas"] == "sawah").to_numpy().astype(int)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
    print("data latih: X_train", Xtr.shape, "+ X_test", Xte.shape, "(0=non_sawah, 1=sawah)")
    print("sebaran train:", pd.Series(ytr).value_counts().sort_index().to_dict())
    print("sebaran test :", pd.Series(yte).value_counts().sort_index().to_dict())
    print("model: RandomForestClassifier(n_estimators=300, random_state=42)")
    clf = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)
    clf.fit(Xtr, ytr)
    pred = clf.predict(Xte)
    cm = confusion_matrix(yte, pred)
    print("confusion matrix (baris=aktual, kolom=prediksi [non_sawah, sawah]):")
    print(cm)
    print(classification_report(yte, pred, target_names=["non_sawah", "sawah"]))
    metrics = {"accuracy": accuracy_score(yte, pred),
               "precision_sawah": precision_score(yte, pred, zero_division=0),
               "recall_sawah": recall_score(yte, pred, zero_division=0),
               "f1_sawah": f1_score(yte, pred, zero_division=0)}
    print(metrics)
    cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
    s = cross_val_score(clf, X, y, cv=cv)
    print("cv5 accuracy:", s.round(3).tolist(), "mean:", round(float(s.mean()), 4))
    print("feature importance (band_1..6 = B02,B03,B04,B08,B11,B12):", clf.feature_importances_.round(3).tolist())
    pd.DataFrame([metrics]).to_csv(OUT + "/jabon_metrics.csv", index=False)
    import joblib
    joblib.dump(clf, OUT + "/jabon_rf.joblib")
    print("tersimpan: jabon_metrics.csv + jabon_rf.joblib")


data latih: X_train (80, 6) + X_test (20, 6) (0=non_sawah, 1=sawah)
sebaran train: {0: 40, 1: 40}
sebaran test : {0: 10, 1: 10}
model: RandomForestClassifier(n_estimators=300, random_state=42)


confusion matrix (baris=aktual, kolom=prediksi [non_sawah, sawah]):
[[10  0]
 [ 0 10]]
              precision    recall  f1-score   support

   non_sawah       1.00      1.00      1.00        10
       sawah       1.00      1.00      1.00        10

    accuracy                           1.00        20
   macro avg       1.00      1.00      1.00        20
weighted avg       1.00      1.00      1.00        20

{'accuracy': 1.0, 'precision_sawah': 1.0, 'recall_sawah': 1.0, 'f1_sawah': 1.0}


cv5 accuracy: [1.0, 1.0, 1.0, 1.0, 0.95] mean: 0.99
feature importance (band_1..6 = B02,B03,B04,B08,B11,B12): [0.122, 0.135, 0.106, 0.201, 0.209, 0.226]


tersimpan: jabon_metrics.csv + jabon_rf.joblib


## Peta hasil klasifikasi

Raster Sentinel-2 → fitur per piksel → prediksi model → `jabon_classified.tif` (1 = sawah, 0 = non-sawah, 255 = nodata) + pratinjau PNG. Ini **peta prediksi biner**, berbeda dari peta 4-kelas `klasifikasi_jabon.tif` (sawah/vegetasi/terbangun/air).

In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

if df_feat is None:
    print("STATUS: BELUM TERSEDIA — peta dilewati.")
else:
    import joblib
    clf = joblib.load(OUT + '/jabon_rf.joblib')
    src = rasterio.open(RASTER)
    arr = src.read().astype(float)
    nodata = src.nodata
    h, w = arr.shape[1], arr.shape[2]
    flat = arr.reshape(arr.shape[0], -1).T
    ok = np.isfinite(flat).all(axis=1) & (~(flat == nodata).any(axis=1))
    out = np.full(flat.shape[0], 255, dtype=np.uint8)
    out[ok] = clf.predict(flat[ok]).astype(np.uint8)
    print("piksel valid: {} dari {}; fraksi sawah (valid): {:.3f}".format(int(ok.sum()), flat.shape[0], float((out[ok] == 1).mean())))
    meta = src.meta.copy()
    meta.update(count=1, dtype="uint8", nodata=255)
    rasterio.open(OUT + "/jabon_classified.tif", "w", **meta).write(out.reshape(h, w), 1)
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(np.ma.masked_equal(out.reshape(h, w), 255), vmin=0, vmax=1)
    ax.set_facecolor('black')
    ax.set_title("Prediksi sawah (1) / non-sawah (0) — Kec. Jabon")
    fig.tight_layout()
    fig.savefig(IMG + 'fig_jabon_preview.png', dpi=100)
    plt.close(fig)
    print("tersimpan: jabon_classified.tif + fig_jabon_preview.png")


piksel valid: 840049 dari 1828233; fraksi sawah (valid): 0.547


tersimpan: jabon_classified.tif + fig_jabon_preview.png


### Peta prediksi (pratinjau)

![Peta prediksi sawah](../data/images/eksperimenjabon/fig_jabon_preview.png)


## Visualisasi EDA fitur (6 gambar)

Boxplot per band per kelas, tanda tangan spektral mean±std, histogram, korelasi antar band, PCA 2D, dan sebaran NDVI per kelas. File `fig_jabon_boxplot.png` s.d. `fig_jabon_ndvi_violin.png` di `data/images/eksperimenjabon/`.

In [5]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

BANDS = ['B02', 'B03', 'B04', 'B08', 'B11', 'B12']
BCOLS = ['band_1', 'band_2', 'band_3', 'band_4', 'band_5', 'band_6']

if df_feat is None:
    print('STATUS: BELUM TERSEDIA — visualisasi EDA dilewati.')
else:
    sw = df_feat[df_feat['kelas'] == 'sawah']
    ns = df_feat[df_feat['kelas'] == 'non_sawah']
    fig, axes = plt.subplots(2, 3, figsize=(12, 7))
    for ax, b, nama in zip(axes.flat, BCOLS, BANDS):
        ax.boxplot([ns[b].to_numpy(), sw[b].to_numpy()])
        ax.set_xticklabels(['non_sawah', 'sawah'])
        ax.set_title(nama)
    fig.suptitle('Sebaran nilai band per kelas (rata-rata poligon, DN Sentinel-2)')
    fig.tight_layout()
    fig.savefig(IMG + 'fig_jabon_boxplot.png', dpi=100)
    plt.close(fig)
    fig, ax = plt.subplots(figsize=(9, 5))
    for g, lab, mk in [(sw, 'sawah', 'o'), (ns, 'non_sawah', 's')]:
        m = g[BCOLS].mean().to_numpy()
        s = g[BCOLS].std().to_numpy()
        ax.errorbar(BANDS, m, yerr=s, marker=mk, capsize=4, label=lab)
    ax.set_title('Tanda tangan spektral rata-rata per kelas')
    ax.set_ylabel('DN rata-rata poligon')
    ax.legend()
    fig.tight_layout()
    fig.savefig(IMG + 'fig_jabon_spektral.png', dpi=100)
    plt.close(fig)
    fig, axes = plt.subplots(2, 3, figsize=(12, 7))
    for ax, b, nama in zip(axes.flat, BCOLS, BANDS):
        ax.hist(ns[b], bins=15, alpha=0.6, label='non_sawah')
        ax.hist(sw[b], bins=15, alpha=0.6, label='sawah')
        ax.set_title(nama)
    axes.flat[0].legend()
    fig.suptitle('Histogram nilai band per kelas')
    fig.tight_layout()
    fig.savefig(IMG + 'fig_jabon_hist.png', dpi=100)
    plt.close(fig)
    corr = df_feat[BCOLS].corr().to_numpy()
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(corr, vmin=-1, vmax=1)
    ax.set_xticks(range(6), BANDS)
    ax.set_yticks(range(6), BANDS)
    for i in range(6):
        for j in range(6):
            ax.text(j, i, '{:.2f}'.format(corr[i, j]), ha='center', va='center', fontsize=8)
    ax.set_title('Korelasi antar band (semua sampel)')
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(IMG + 'fig_jabon_korelasi.png', dpi=100)
    plt.close(fig)
    Xs = StandardScaler().fit_transform(df_feat[BCOLS].to_numpy())
    P = PCA(n_components=2, random_state=SEED).fit_transform(Xs)
    print('PCA explained variance:', PCA(n_components=2, random_state=SEED).fit(Xs).explained_variance_ratio_.round(3).tolist())
    fig, ax = plt.subplots(figsize=(6, 5))
    m1 = df_feat['kelas'].to_numpy() == 'non_sawah'
    ax.scatter(P[m1, 0], P[m1, 1], alpha=0.7, label='non_sawah')
    ax.scatter(P[~m1, 0], P[~m1, 1], alpha=0.7, label='sawah')
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.set_title('PCA 2D fitur band per poligon')
    ax.legend()
    fig.tight_layout()
    fig.savefig(IMG + 'fig_jabon_pca.png', dpi=100)
    plt.close(fig)
    ndvi_sw = (sw['band_4'] - sw['band_3']) / (sw['band_4'] + sw['band_3'])
    ndvi_ns = (ns['band_4'] - ns['band_3']) / (ns['band_4'] + ns['band_3'])
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.violinplot([ndvi_ns.to_numpy(), ndvi_sw.to_numpy()], showmeans=True)
    ax.set_xticks([1, 2], ['non_sawah', 'sawah'])
    ax.set_title('Sebaran NDVI per kelas (dari rata-rata poligon)')
    fig.tight_layout()
    fig.savefig(IMG + 'fig_jabon_ndvi_violin.png', dpi=100)
    plt.close(fig)
    print('tersimpan: 6 gambar EDA jabon')


PCA explained variance: [0.859, 0.113]


tersimpan: 6 gambar EDA jabon


### Galeri EDA fitur

![Boxplot band per kelas](../data/images/eksperimenjabon/fig_jabon_boxplot.png)

![Tanda tangan spektral](../data/images/eksperimenjabon/fig_jabon_spektral.png)

![Histogram band](../data/images/eksperimenjabon/fig_jabon_hist.png)

![Korelasi band](../data/images/eksperimenjabon/fig_jabon_korelasi.png)

![PCA 2D](../data/images/eksperimenjabon/fig_jabon_pca.png)

![NDVI per kelas](../data/images/eksperimenjabon/fig_jabon_ndvi_violin.png)


## Visualisasi evaluasi dan peta (6 gambar)

Confusion matrix, feature importance, komposit RGB + overlay 100 poligon sampel, peta probabilitas sawah, banding peta 4-kelas vs prediksi biner, dan peta NDVI Jabon.

In [6]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix as cm_fn
import geopandas as gpd
import rasterio

if df_feat is None:
    print('STATUS: BELUM TERSEDIA — visualisasi evaluasi/peta dilewati.')
else:
    import joblib
    clf2 = joblib.load(OUT + '/jabon_rf.joblib')
    X = df_feat.filter(like='band_').to_numpy()
    y = (df_feat['kelas'] == 'sawah').to_numpy().astype(int)
    _, Xte, _, yte = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
    pred = clf2.predict(Xte)
    cm = cm_fn(yte, pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm)
    ax.set_xticks([0, 1], ['pred non_sawah', 'pred sawah'])
    ax.set_yticks([0, 1], ['aktual non_sawah', 'aktual sawah'])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=14)
    ax.set_title('Confusion matrix (test 20 sampel)')
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(IMG + 'fig_jabon_cm.png', dpi=100)
    plt.close(fig)
    imp = clf2.feature_importances_
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.barh(BANDS, imp)
    ax.set_xlabel('importance')
    ax.set_title('Feature importance RandomForest')
    for i, v in enumerate(imp):
        ax.text(v, i, ' {:.3f}'.format(v), va='center')
    fig.tight_layout()
    fig.savefig(IMG + 'fig_jabon_importance.png', dpi=100)
    plt.close(fig)
    src = rasterio.open(RASTER)
    arr6 = src.read().astype(float)
    ok6 = np.isfinite(arr6).all(axis=0) & (~(arr6 == src.nodata).any(axis=0))
    rgb = np.stack([arr6[2], arr6[1], arr6[0]])
    lo, hi = np.percentile(rgb[:, ok6], [2, 98])
    rgb8 = np.clip((rgb - lo) / (hi - lo), 0, 1)
    b = src.bounds
    shp2 = gpd.read_file(SHP)
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(np.moveaxis(rgb8, 0, -1), extent=(b.left, b.right, b.bottom, b.top))
    shp2[shp2['kelas'] == 'sawah'].plot(ax=ax, facecolor='none', edgecolor='lime', linewidth=0.8, label='sawah')
    shp2[shp2['kelas'] == 'non_sawah'].plot(ax=ax, facecolor='none', edgecolor='red', linewidth=0.8, label='non_sawah')
    ax.set_title('Komposit RGB (B04/B03/B02) + 100 poligon sampel')
    ax.legend()
    fig.tight_layout()
    fig.savefig(IMG + 'fig_jabon_rgb_sampel.png', dpi=100)
    plt.close(fig)
    flat = arr6.reshape(6, -1).T
    okf = np.isfinite(flat).all(axis=1) & (~(flat == src.nodata).any(axis=1))
    proba = np.full(flat.shape[0], np.nan)
    proba[okf] = clf2.predict_proba(flat[okf])[:, 1]
    fig, ax = plt.subplots(figsize=(7, 7))
    im = ax.imshow(proba.reshape(arr6.shape[1], arr6.shape[2]), vmin=0, vmax=1)
    ax.set_title('Probabilitas sawah (RandomForest)')
    fig.colorbar(im, ax=ax, label='P(sawah)')
    fig.tight_layout()
    fig.savefig(IMG + 'fig_jabon_prob.png', dpi=100)
    plt.close(fig)
    src.close()
    k4 = rasterio.open('../data/digitasi_kamal/klasifikasi_jabon.tif').read(1)
    binmap = rasterio.open(OUT + '/jabon_classified.tif').read(1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 6))
    cmap4 = ListedColormap(['black', 'green', 'darkgreen', 'gray', 'blue'])
    axes[0].imshow(k4, vmin=0, vmax=4, cmap=cmap4)
    axes[0].set_title('Acuan 4-kelas (0=nodata,1=sawah,2=veg,3=bangun,4=air)')
    binm = np.ma.masked_equal(binmap, 255)
    cmap_bin = plt.cm.viridis.copy()
    cmap_bin.set_bad('black')
    axes[1].imshow(binm, vmin=0, vmax=1, cmap=cmap_bin)
    axes[1].set_title('Prediksi biner RF (1=sawah, 0=non-sawah)')
    fig.suptitle('Banding peta acuan vs prediksi')
    fig.tight_layout()
    fig.savefig(IMG + 'fig_jabon_banding.png', dpi=100)
    plt.close(fig)
    b04 = arr6[2]
    b08 = arr6[3]
    with np.errstate(divide='ignore', invalid='ignore'):
        ndvi_map = (b08 - b04) / (b08 + b04)
    ndvi_map[~ok6] = np.nan
    fig, ax = plt.subplots(figsize=(7, 7))
    im = ax.imshow(ndvi_map, vmin=-1, vmax=1, cmap='RdYlGn')
    ax.set_title('NDVI Jabon (dari B08/B04)')
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(IMG + 'fig_jabon_ndvi_peta.png', dpi=100)
    plt.close(fig)
    print('tersimpan: 6 gambar evaluasi/peta jabon')


tersimpan: 6 gambar evaluasi/peta jabon


### Galeri evaluasi dan peta

![Confusion matrix](../data/images/eksperimenjabon/fig_jabon_cm.png)

![Feature importance](../data/images/eksperimenjabon/fig_jabon_importance.png)

![RGB + sampel](../data/images/eksperimenjabon/fig_jabon_rgb_sampel.png)

![Probabilitas sawah](../data/images/eksperimenjabon/fig_jabon_prob.png)

![Banding acuan vs prediksi](../data/images/eksperimenjabon/fig_jabon_banding.png)

![Peta NDVI](../data/images/eksperimenjabon/fig_jabon_ndvi_peta.png)


## Unduh Berkas Klasifikasi Sawah Jabon

*   **jabon_features.csv** — Fitur mean 6 band 100 poligon (50 sawah + 50 non-sawah):  
    <a href="../data/processed/jabon_features.csv" download style="display:inline-block;padding:8px 18px;margin:4px 6px 4px 0;background:#1a73e8;color:#ffffff;text-decoration:none;border-radius:8px;font-weight:bold;font-size:14px;">Unduh jabon_features.csv</a>

*   **jabon_metrics.csv** — Metrik evaluasi (accuracy/precision/recall/F1):  
    <a href="../data/processed/jabon_metrics.csv" download style="display:inline-block;padding:8px 18px;margin:4px 6px 4px 0;background:#1a73e8;color:#ffffff;text-decoration:none;border-radius:8px;font-weight:bold;font-size:14px;">Unduh jabon_metrics.csv</a>

*   **jabon_classified.tif** — Raster biner (1=sawah, 0=non-sawah, 255=nodata):  
    <a href="../data/processed/jabon_classified.tif" download style="display:inline-block;padding:8px 18px;margin:4px 6px 4px 0;background:#1a73e8;color:#ffffff;text-decoration:none;border-radius:8px;font-weight:bold;font-size:14px;">Unduh jabon_classified.tif</a>

*   **jabon_rf.joblib** — Model RandomForest terlatih (300 pohon):  
    <a href="../data/processed/jabon_rf.joblib" download style="display:inline-block;padding:8px 18px;margin:4px 6px 4px 0;background:#1a73e8;color:#ffffff;text-decoration:none;border-radius:8px;font-weight:bold;font-size:14px;">Unduh jabon_rf.joblib</a>

*   **jabon_sawah.shp** — 100 poligon sampel (+ sidecar .dbf/.shx/.prj/.cpg):  
    <a href="../data/spasial/jabon_sawah.shp" download style="display:inline-block;padding:8px 18px;margin:4px 6px 4px 0;background:#1a73e8;color:#ffffff;text-decoration:none;border-radius:8px;font-weight:bold;font-size:14px;">Unduh jabon_sawah.shp</a>

*   **s2_jabon.tif** — Komposit Sentinel-2 L2A Jabon 6 band (8,3 MB):  
    <a href="../data/spasial/s2_jabon.tif" download style="display:inline-block;padding:8px 18px;margin:4px 6px 4px 0;background:#1a73e8;color:#ffffff;text-decoration:none;border-radius:8px;font-weight:bold;font-size:14px;">Unduh s2_jabon.tif</a>


## Status akhir 6.1 — SELESAI (Jabon)

- Training: 100 poligon (50 sawah + 50 non_sawah; non = pohon/kebun 25 + terbangun 25); test 20 sampel (10+10): confusion matrix [[10, 0], [0, 10]], accuracy/precision/recall/F1 = 1.0; CV 5-fold [1.0, 1.0, 1.0, 1.0, 0.95] mean 0.99. Nilai tinggi ini SIRCULAR (label kandidat dari raster yang sama) — satu fold yang tidak sempurna (0.95) justru lebih jujur daripada skor mutlak.
- Feature importance: B12 (0.226) > B11 (0.209) > B08 (0.201) > B03 > B02 > B04 — SWIR+NIR dominan, masuk akal fisis untuk memisahkan lahan bera/vegetasi/terbangun.
- Peta: `data/processed/jabon_classified.tif` (1 = sawah, 0 = non-sawah, 255 = nodata); 840.049 piksel valid (45,9%), fraksi sawah 0,547. Pembanding acuan 4-kelas (`klasifikasi_jabon.tif`): sawah 26,0% — model biner OVER-prediksi sawah ~29 poin karena air/tambak (26,6% area, tak disampel) ikut terlabel sawah. Jangan pakai angka luas biner ini tanpa koreksi.
- File: `jabon_features.csv` (100 baris), `jabon_metrics.csv`, `jabon_rf.joblib`, `jabon_classified.tif`, `data/spasial/jabon_sawah.shp`, 13 PNG (`fig_jabon_preview.png` + 12 EDA/evaluasi).
- Arti FP/FN: false positive = non-sawah diprediksi sawah (over-estimasi luas); false negative = sawah tak terdeteksi (under-estimasi).
- Syarat pakai serius: validasi lapangan/citra resolusi tinggi (khususnya bedakan tambak vs sawah) + tambah sampel air/tambak + split blok spasial, lalu latih ulang.